The following is an example of how to utilize our Sen1Floods11 dataset for training a FCNN. In this example, we train and validate on hand-labeled chips of flood events. However, our dataset includes several other options that are detailed in the README. To replace the dataset, as outlined further below, simply replace the train, test, and validation split csv's, and download the corresponding dataset.

Authenticate Google Cloud Platform. Note that to run this code, you must connect your notebook runtime to a GPU.

In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu130
True
Tesla T4


Install RasterIO

In [1]:
!pip install rasterio

Define a model checkpoint folder, for storing network checkpoints during training

Download train, test, and validation splits for both flood water. To download different train, test, and validation splits, simply replace these paths with the path to a csv containing the desired splits.

In [3]:
import torch
import torchvision
import rasterio
import numpy as np

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Rasterio:", rasterio.__version__)

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_train_data.csv...
/ [1 files][ 13.3 KiB/ 13.3 KiB]                                                
Operation completed over 1 objects/13.3 KiB.                                     
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv...
/ [1 files][  4.7 KiB/  4.7 KiB]                                                
Operation completed over 1 objects/4.7 KiB.                                    

Download raw train, test, and validation data. In this example, we are downloading train, test, and validation data of flood images which are hand labeled. However, you can simply replace these paths with whichever dataset you would like to use - further documentation of the Sen1Floods11 dataset and organization is available in the README.

In [ ]:
from pathlib import Path

ROOT = Path("/content/sen1floods11")
S1_DIR = ROOT / "S1"
LABEL_DIR = ROOT / "Labels"
SPLIT_DIR = ROOT / "splits"
CHECKPOINT_DIR = ROOT / "checkpoints"

for path in [S1_DIR, LABEL_DIR, SPLIT_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.

both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Building synchronization state...
Starting synchronization...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_314919_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_23014_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_103757_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_12933

Define model training hyperparameters

In [ ]:
!gsutil cp \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_train_data.csv \
  /content/sen1floods11/splits/

!gsutil cp \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_valid_data.csv \
  /content/sen1floods11/splits/

!gsutil cp \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv \
  /content/sen1floods11/splits/

Define functions to process and augment training and testing images

In [ ]:
!gsutil -m rsync -r \
  gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand \
  /content/sen1floods11/S1

!gsutil -m rsync -r \
  gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand \
  /content/sen1floods11/Labels

In [ ]:
print("S1:", len(list(S1_DIR.glob("*.tif"))))
print("Labels:", len(list(LABEL_DIR.glob("*.tif"))))
print("Splits:", list(SPLIT_DIR.glob("*.csv")))

Load *flood water* train, test, and validation data from splits. In this example, this is the data we will use to train our model.

In [ ]:
import rasterio
import numpy as np

sample_path = next(S1_DIR.glob("*.tif"))

with rasterio.open(sample_path) as src:
    image = src.read()
    print("shape:", image.shape)
    print("dtype:", image.dtype)
    print("CRS:", src.crs)
    print("范围:", np.nanmin(image), np.nanmax(image))

Load training data and validation data. Note that here, we have chosen to train and validate our model on flood data. However, you can simply replace the load function call with one of the options defined above to load a different dataset.

In [ ]:
import csv
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

class Sen1Floods11Dataset(Dataset):
    def __init__(self, split_csv, s1_dir, label_dir, training=False):
        self.s1_dir = Path(s1_dir)
        self.label_dir = Path(label_dir)
        self.training = training
        self.items = []

        with open(split_csv, newline="") as f:
            for row in csv.reader(f):
                s1_name = Path(row[0]).name
                label_name = Path(row[1]).name
                self.items.append((s1_name, label_name))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        s1_name, label_name = self.items[index]

        with rasterio.open(self.s1_dir / s1_name) as src:
            image = src.read([1, 2]).astype(np.float32)

        with rasterio.open(self.label_dir / label_name) as src:
            mask = src.read(1).astype(np.int64)

        # 与官方Notebook一致
        image = np.nan_to_num(
            image,
            nan=0.0,
            posinf=1.0,
            neginf=-50.0,
        )
        image = np.clip(image, -50.0, 1.0)
        image = (image + 50.0) / 51.0

        mask[mask == -1] = 255

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)

        if self.training:
            top = random.randint(0, image.shape[1] - 256)
            left = random.randint(0, image.shape[2] - 256)

            image = image[:, top:top + 256, left:left + 256]
            mask = mask[top:top + 256, left:left + 256]

            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            if random.random() < 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask)
        else:
            image = TF.resize(
                image,
                [512, 512],
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            mask = TF.resize(
                mask.unsqueeze(0),
                [512, 512],
                interpolation=InterpolationMode.NEAREST,
            ).squeeze(0)

        # 官方数据集统计量
        mean = torch.tensor([0.6851, 0.5235])[:, None, None]
        std = torch.tensor([0.0820, 0.1102])[:, None, None]
        image = (image - mean) / std

        return image.float(), mask.long()

Define the network. For our purposes, we use ResNet50. However, if you wish to test a different model framework, optimizer, or loss function you can simply replace those here.

In [ ]:
train_dataset = Sen1Floods11Dataset(
    SPLIT_DIR / "flood_train_data.csv",
    S1_DIR,
    LABEL_DIR,
    training=True,
)

valid_dataset = Sen1Floods11Dataset(
    SPLIT_DIR / "flood_valid_data.csv",
    S1_DIR,
    LABEL_DIR,
    training=False,
)

test_dataset = Sen1Floods11Dataset(
    SPLIT_DIR / "flood_test_data.csv",
    S1_DIR,
    LABEL_DIR,
    training=False,
)

Define assessment metrics. For our purposes, we use overall accuracy and mean intersection over union. However, we also include functions for calculating true positives, false positives, true negatives, and false negatives.

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
images, masks = next(iter(train_loader))

print(images.shape)
print(masks.shape)
print(torch.unique(masks))

Define training loop

In [ ]:
import torch.nn as nn
from torchvision.models.segmentation import fcn_resnet50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = fcn_resnet50(
    weights=None,
    weights_backbone=None,
    num_classes=2,
)

# 将RGB三通道输入改成VV/VH双通道
model.backbone.conv1 = nn.Conv2d(
    2,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False,
)

Define validation loop

In [ ]:
def replace_bn_with_gn(module, num_groups=16):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            groups = min(num_groups, child.num_features)

            while child.num_features % groups != 0:
                groups -= 1

            setattr(
                module,
                name,
                nn.GroupNorm(groups, child.num_features),
            )
        else:
            replace_bn_with_gn(child, num_groups)

replace_bn_with_gn(model)
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 8.0], device=device),
    ignore_index=255,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=len(train_loader) * 10,
    T_mult=2,
)

Define testing loop (here, you can replace assessment metrics).

In [ ]:
from tqdm.auto import tqdm

def run_epoch(model, loader, training):
    model.train(training)

    loss_sum = 0.0
    intersection = 0
    union = 0

    for images, masks in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(images)["out"]
            loss = criterion(logits, masks)

            if training:
                loss.backward()
                optimizer.step()
                scheduler.step()

        prediction = logits.argmax(dim=1)
        valid = masks != 255
        water_pred = (prediction == 1) & valid
        water_true = (masks == 1) & valid

        intersection += (water_pred & water_true).sum().item()
        union += (water_pred | water_true).sum().item()
        loss_sum += loss.item()

    return {
        "loss": loss_sum / len(loader),
        "water_iou": intersection / max(union, 1),
    }

In [ ]:
EPOCHS = 2
best_iou = -1.0

for epoch in range(EPOCHS):
    train_metrics = run_epoch(model, train_loader, training=True)
    valid_metrics = run_epoch(model, valid_loader, training=False)

    print(
        f"epoch={epoch + 1} "
        f"train_loss={train_metrics['loss']:.4f} "
        f"train_iou={train_metrics['water_iou']:.4f} "
        f"valid_loss={valid_metrics['loss']:.4f} "
        f"valid_iou={valid_metrics['water_iou']:.4f}"
    )

    if valid_metrics["water_iou"] > best_iou:
        best_iou = valid_metrics["water_iou"]

        torch.save(
            {
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "best_iou": best_iou,
            },
            CHECKPOINT_DIR / "best_fcn_resnet50.pt",
        )

Define training and validation scheme

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_DIR / "best_fcn_resnet50.pt",
    map_location=device,
)

model.load_state_dict(checkpoint["model"])

test_metrics = run_epoch(
    model,
    test_loader,
    training=False,
)

print(test_metrics)